# Reviewer 3: full-generator audit and figure revision

This **new revision analysis** audits the existing numerical parity closure; it does not replace it by an inferred crystallographic sublattice model. The inherited transport law is embedded verbatim and checked against the tridiagonal implementation. Fixed background profiles are numerical probes, not measured hydrogen profiles. The propagated state is a normalized probability under a frozen generator, not a self-consistent nonlinear concentration trajectory.

The two tests retain the complete finite-mesh site energy and edge diffusivity. They compare with the expressly identified smooth limiting coefficient problem (mesh-derived staggered amplitude tends to zero). A nonzero physical crystallographic order is not identified by this limit. A grid-alternating counterexample records why simply retaining a constant alternating amplitude changes the resolved physical length scale.

The original manuscript's smooth constant-field numbers are retained separately. The newly observed first-order gradient result is reported without imposing a second-order pass criterion. The stationary component is propagated exactly; 128-mode and 256-mode calculations, direct matrix-exponential checks, fine-reference doubling, conservation, and detailed balance are audited. All declared parameter values and source provenance are embedded below.

The plotting cell redraws inherited polarization/structural/risk data and new transport-test results into seven separate single-axis figures. The biexponential overlay is from the already supplied retrospective revision analysis, not a new original holdout. Structural/risk/hidden-twin panels retain the inherited model assumptions. PDFs, editable SVGs, and numerical tables are written to `proton_R3_outputs`. No online access or original notebook execution is required.


In [1]:
"""New frozen-generator refinement audit; no changes to inherited model fits."""
from pathlib import Path
import json,ast,hashlib
import numpy as np
import pandas as pd
from scipy.linalg import eigh_tridiagonal
from scipy.sparse import diags
from scipy.sparse.linalg import expm_multiply
BASE=Path.cwd() / 'proton_R3_outputs';BASE.mkdir(exist_ok=True)
OUT=BASE/'new_results';OUT.mkdir(exist_ok=True)
s={'KB_EV_K': 8.617333262145179e-05, 'E_A_RELAX_EV': 0.17403869463781, 'E_R_RELAX_EV_FU': 0.066, 'D_REP_M2_S': 1.3416407864998738e-18, 'TRAP_FIT': [-7.262406628943365, 0.9698094585796684, -0.3065490977982868, -0.12385171386161944, 2.6814065218852408, 1.3412868590757983], 'Q_A_TARGET_A': 0.18313538820706152, 'EA_MODEL_EV': 0.41, 'Q_R_COUPLED': 0.000721655362771062, 'COUPLED_GAP': 4.3518042730445334e-05}
kb=s['KB_EV_K'];EA=s['E_A_RELAX_EV'];ER=s['E_R_RELAX_EV_FU'];Drep=s['D_REP_M2_S'];chi=s['TRAP_FIT'][1]
T=300.;beta=1/(kb*T);L=100e-9;E=2e5;strain=.0
TIME=.08*L**2/Drep

def background(n,case):
 y=(np.arange(n)+.5)/n
 if case=='symmetric':c=.3+.1*np.cos(2*np.pi*y)
 elif case=='ramp':c=.2+.2*y
 else:raise ValueError(case)
 return y,c

def rates(n,case,limiting=False):
 y,c=background(n,case);sign=(-1.)**np.arange(n)
 m=abs(sign@c)/c.sum()
 if limiting:m=0. # explicitly the smooth h->0 reference, not a finite-mesh replacement
 U=-E*L*y-.2*EA*c-.1*ER*m*sign
 De=Drep*np.exp(-beta*(chi*EA*(c[:-1]+c[1:])/2+.2*ER*m+.15*EA*strain/.01))
 h=L/n;du=np.diff(U);kf=De/h**2*np.exp(-beta*du/2);kr=De/h**2*np.exp(beta*du/2)
 diag=np.zeros(n);diag[:-1]-=kf;diag[1:]-=kr
 w=np.exp(-beta*U);pi=w/w.sum()
 return y,c,U,De,kf,kr,diag,pi,m

def evolve(n,case,limiting=False,modes=128):
 y,c,U,De,kf,kr,diag,pi,m=rates(n,case,limiting)
 lam,V=eigh_tridiagonal(-diag,-np.sqrt(kf*kr),select='i',select_range=(0,min(modes,n)-1),lapack_driver='stebz',tol=1e-16)
 # Standard similarity S=diag(pi)^(-1/2)Q diag(pi)^(1/2).
 p0=(1+.2*np.sinc(.5/n)*np.cos(np.pi*y))/n
 sqrtpi=np.sqrt(pi)
 # Treat the exactly known stationary component analytically, avoiding numerical
 # drift from the computed near-zero eigenvalue.
 p=pi+sqrtpi*(V[:,1:]@(np.exp(-lam[1:]*TIME)*(V[:,1:].T@((p0-pi)/sqrtpi))))
 # entropy derivative at a prescribed positive non-equilibrium probability probe
 probe=.15+.7*np.exp(-((y-.35)/.18)**2);probe/=probe.sum()
 dp=diag*probe;dp[1:]+=kf*probe[:-1];dp[:-1]+=kr*probe[1:]
 Hdot=float(dp@np.log(probe/pi))
 B=4*De.min()/(L/n)**2*np.sin(np.pi/(2*n))**2*np.exp(-beta*np.ptp(U))
 db=np.max(np.abs(np.log(kf/kr)+beta*np.diff(U)))
 mass_error=abs(p.sum()-1)
 Q=diags([kr,diag,kf],[1,0,-1],format='csc')
 relcol=max(abs(np.asarray(Q.sum(axis=0)).ravel()))/max(abs(diag))
 result=dict(case=case,N=n,Q_R_A=.06*m,gap_s_1=lam[1],lower_bound_s_1=B,gap_ratio=lam[1]/B,
 entropy_derivative_s_1=Hdot,D_min=De.min(),D_max=De.max(),U_osc_eV=np.ptp(U),
 db_residual=db,relative_column_residual=relcol,mass_error=mass_error)
 return p,result

rows=[];probes={};refs={};ref_checks=[]
for case in ['symmetric','ramp']:
 print('REFERENCE',case,flush=True)
 pref,_=evolve(4096,case,limiting=True)
 pref2,_=evolve(8192,case,limiting=True)
 pref_more,_=evolve(4096,case,limiting=True,modes=256)
 ref_checks.append(dict(case=case,reference_doubling_L2=float(np.linalg.norm(pref2.reshape(4096,2).sum(1)-pref)/np.linalg.norm(pref)),
  modal_truncation_L2=float(np.linalg.norm(pref-pref_more)/np.linalg.norm(pref))))
 refs[case]=pref2
 for n in [16,32,64,128,256,512,1024]:
  pn,row=evolve(n,case)
  refavg=pref2.reshape(n,-1).sum(1)
  row['relative_L2_error']=float(np.linalg.norm(pn-refavg)/np.linalg.norm(refavg))
  # Same finite full law at 8192 as an additional refinement comparator.
  probes[case+str(n)]=pn
  rows.append(row)

 # verify modal propagation vs directly computed matrix exponential for n=64
 p64,_=evolve(64,case)
 y,c,U,D,kf,kr,d,pi,m=rates(64,case)
 pp=expm_multiply(diags([kr,d,kf],[1,0,-1],format='csc')*TIME,(1+.2*np.sinc(.5/64)*np.cos(np.pi*y))/64)
 ref_checks[-1]['matrix_exponential_replay_L2']=float(np.linalg.norm(pp-p64)/np.linalg.norm(pp))

df=pd.DataFrame(rows)
orders=[]
for case,g in df.groupby('case',sort=False):
 orders.append(dict(case=case,all_grid_order=float(np.polyfit(np.log(1/g.N),np.log(g.relative_L2_error),1)[0]),
 fine_grid_order=float(np.polyfit(np.log(1/g.N.iloc[-4:]),np.log(g.relative_L2_error.iloc[-4:]),1)[0])))
# Uniform analytic comparison for all declared mesh sizes N>=16.
Dglobal=Drep*np.exp(-beta*(chi*EA*.4+.2*ER/48))
Uglobal=E*L+.2*EA*.2+2*.1*ER/48
Bglobal=8*Dglobal/L**2*np.exp(-beta*Uglobal)
# Audit the exact generator implementation at N=32.
from types import SimpleNamespace
source='def coupled_local_generator(\n    profile: np.ndarray,\n    temperature_k: float,\n    field_v_m: float,\n    strain: float = 0.0,\n) -> tuple[np.ndarray, np.ndarray, np.ndarray, float, list[tuple[int, int, float, float, float]]]:\n    """Mean-field local detailed-balance generator with explicit structural feedback.\n\n    The local donated-electron occupancy is n_e=c and Q_A,i=Q_A^(1)n_e,i.\n    The edge mobility is reduced by the fitted structural barrier transfer chi_A.\n    A bounded staggered R2 mode and strain shift are included without allowing\n    recoverability to enter the rates or forces.\n    """\n    c = np.clip(np.asarray(profile, dtype=float), 1e-12, 1.0 - 1e-12)\n    n = len(c)\n    h = CFG.length_m / n\n    x = (np.arange(n) + 0.5) * h\n    n_e = c.copy()\n    q_a = Q_A_TARGET_A * n_e\n    alternating = (-1.0) ** np.arange(n)\n    m_pi = float(abs(alternating @ n_e) / max(n_e.sum(), 1e-15))\n    q_r = float(Q_R_BULK_A * m_pi)\n\n    d_reference = D_REP_M2_S * np.exp(-EA_MODEL_EV / KB_EV_K * (1.0 / temperature_k - 1.0 / T_REF_K))\n    beta = 1.0 / (KB_EV_K * temperature_k)\n    # A weak site-energy contribution represents local electron-lattice trapping.\n    u_ev = (\n        -field_v_m * x\n        - 0.20 * E_A_RELAX_EV * (q_a / Q_A_TARGET_A)\n        - 0.10 * E_R_RELAX_EV_FU * (q_r / Q_R_BULK_A if q_r > 0.0 else 0.0) * alternating\n    )\n    q_matrix = np.zeros((n, n), dtype=float)\n    edge_records = []\n    for i in range(n - 1):\n        j = i + 1\n        q_edge_fraction = 0.5 * (q_a[i] + q_a[j]) / Q_A_TARGET_A\n        barrier_shift_ev = (\n            TRAP_FIT.x[1] * E_A_RELAX_EV * q_edge_fraction\n            + 0.20 * E_R_RELAX_EV_FU * (q_r / Q_R_BULK_A if q_r > 0.0 else 0.0)\n            + 0.15 * E_A_RELAX_EV * strain / 0.01\n        )\n        edge_diffusivity = d_reference * np.exp(-barrier_shift_ev * beta)\n        k0 = edge_diffusivity / h**2\n        du = u_ev[j] - u_ev[i]\n        k_i_j = k0 * np.exp(-0.5 * beta * du)\n        k_j_i = k0 * np.exp(+0.5 * beta * du)\n        q_matrix[j, i] += k_i_j\n        q_matrix[i, i] -= k_i_j\n        q_matrix[i, j] += k_j_i\n        q_matrix[j, j] -= k_j_i\n        edge_records.append((i, j, k_i_j, k_j_i, du))\n    return q_matrix, q_a, u_ev, q_r, edge_records';tree=ast.parse(source)
node=next(x for x in tree.body if isinstance(x,ast.FunctionDef) and x.name=='coupled_local_generator')
ns={'np':np,'CFG':SimpleNamespace(length_m=L),'Q_A_TARGET_A':s['Q_A_TARGET_A'],'Q_R_BULK_A':.06,
 'D_REP_M2_S':Drep,'EA_MODEL_EV':s['EA_MODEL_EV'],'KB_EV_K':kb,'T_REF_K':300.,
 'E_A_RELAX_EV':EA,'E_R_RELAX_EV_FU':ER,'TRAP_FIT':SimpleNamespace(x=np.asarray(s['TRAP_FIT']))}
exec(compile(ast.Module(body=[node],type_ignores=[]),'<verbatim original generator>','exec'),ns)
for case in ['symmetric','ramp']:
 y,c,U,D,kf,kr,d,pi,m=rates(32,case)
 Q0,qa,U0,qR0,edges=ns['coupled_local_generator'](c,T,E,strain)
 Q=diags([kr,d,kf],[1,0,-1]).toarray()
 ref_checks.append(dict(case=case,original_generator_relative_difference=float(np.max(abs(Q-Q0))/np.max(abs(Q0)))))
 assert np.allclose(Q,Q0,rtol=2e-15,atol=1e-18)
# A nonvanishing grid-alternating state under refinement is not one smooth physical profile.
parity=[]
for n in [16,32,64,128,256,512,1024]:
 c=.3+.05*(-1.)**np.arange(n)
 m=abs((-1.)**np.arange(n)@c)/c.sum()
 parity.append(dict(N=n,m_pi=m,Q_R_A=.06*m,nonvanishing_density_variance=float(np.var(c))))
summary=dict(time_s=TIME,L_m=L,T_K=T,E_V_m=E,chi=chi,orders=orders,
 uniform_bound_s_1=Bglobal,all_mesh_gaps_exceed_uniform_bound=bool((df.gap_s_1>Bglobal).all()),
 max_mass_error=float(df.mass_error.max()),max_db_residual=float(df.db_residual.max()),
 original_full_generator_audit_Q_R_A=s['Q_R_COUPLED'],original_full_generator_audit_gap=s['COUPLED_GAP'],
 fine_reference_cells=8192,reference_definition='Smooth limit m_pi=0, same spatial c-dependent U and D; full finite-N coefficients retained in every test.',
 test_scope='Frozen prescribed occupancy background; evolving normalized probability. Not nonlinear self-consistent transport or repaired crystallographic sublattices.',
 old_notebook_sha256='3929eb8def57f4f883706baf2a099b7ad13c8fd6d6bf6cd793ff83afa03640ad')
df.to_csv(OUT/'full_generator_refinement.csv',index=False)
pd.DataFrame(parity).to_csv(OUT/'grid_alternating_counterexample.csv',index=False)
(OUT/'summary.json').write_text(json.dumps(summary,indent=2))
(OUT/'numerical_checks.json').write_text(json.dumps(ref_checks,indent=2))
np.savez_compressed(OUT/'probability_profiles.npz',**probes,reference_symmetric=refs['symmetric'],reference_ramp=refs['ramp'])
print('SUMMARY',json.dumps(summary,indent=2),flush=True)
print('CHECKS',ref_checks,flush=True)
assert df.gap_s_1.min()>0 and (df.gap_s_1>=df.lower_bound_s_1).all()
assert df.db_residual.max()<1e-12 and (df.entropy_derivative_s_1<0).all()
assert all(r.get('modal_truncation_L2',0)<1e-11 for r in ref_checks)

# Recheck the original selected audit profile with the verbatim generator.
audit_profile=np.array([0.7083660215462788, 0.6914567245690483, 0.6749618223247568, 0.6588709595020285, 0.6431739420834406, 0.6278607407938662, 0.6129214950527526, 0.5983465173281155, 0.5841262977898144, 0.5702515091604623, 0.5567130116640169, 0.5435018579746965, 0.5306092980723272, 0.5180267839144563, 0.5057459738405745, 0.49375873662942826, 0.4820571551366788, 0.47063352944695347, 0.45948037948157017, 0.44859044701083284, 0.4379566970277007, 0.4275723184477315, 0.41743072410844473, 0.40752555004950985, 0.39785065406340836, 0.38840011351433557, 0.37916822243102555, 0.3701494878868581, 0.36133862568792474, 0.35273055539667886, 0.3443203947252774, 0.3361034533387113])
audit_Q,_,audit_U,audit_QR,_=ns["coupled_local_generator"](audit_profile,T,E)
assert abs(audit_QR-s["Q_R_COUPLED"])<1e-15
pd.DataFrame({"cell":np.arange(1,33),"occupancy":audit_profile}).to_csv(OUT/"original_audit_profile.csv",index=False)
print("\nFULL GENERATOR REFINEMENT\n",df[["case","N","Q_R_A","relative_L2_error","gap_s_1","gap_ratio"]].to_string(index=False))


REFERENCE symmetric


REFERENCE ramp


SUMMARY {
  "time_s": 596.2847939999439,
  "L_m": 1e-07,
  "T_K": 300.0,
  "E_V_m": 200000.0,
  "chi": 0.9698094585796684,
  "orders": [
    {
      "case": "symmetric",
      "all_grid_order": 2.0030854112507948,
      "fine_grid_order": 2.007129324352044
    },
    {
      "case": "ramp",
      "all_grid_order": 1.001640068333472,
      "fine_grid_order": 1.0003697143689563
    }
  ],
  "uniform_bound_s_1": 2.718764030000985e-05,
  "all_mesh_gaps_exceed_uniform_bound": true,
  "max_mass_error": 1.085798118083403e-12,
  "max_db_residual": 3.69712940817557e-16,
  "original_full_generator_audit_Q_R_A": 0.000721655362771062,
  "original_full_generator_audit_gap": 4.3518042730445334e-05,
  "fine_reference_cells": 8192,
  "reference_definition": "Smooth limit m_pi=0, same spatial c-dependent U and D; full finite-N coefficients retained in every test.",
  "test_scope": "Frozen prescribed occupancy background; evolving normalized probability. Not nonlinear self-consistent transport or repair

CHECKS [{'case': 'symmetric', 'reference_doubling_L2': 4.03166484089586e-09, 'modal_truncation_L2': 2.139685788596101e-13, 'matrix_exponential_replay_L2': 3.393141839070187e-15}, {'case': 'ramp', 'reference_doubling_L2': 4.746053623762854e-09, 'modal_truncation_L2': 0.0, 'matrix_exponential_replay_L2': 2.6734264458635263e-14}, {'case': 'symmetric', 'original_generator_relative_difference': 3.7577576421699327e-16}, {'case': 'ramp', 'original_generator_relative_difference': 2.483637664455283e-16}]



FULL GENERATOR REFINEMENT
      case    N        Q_R_A  relative_L2_error  gap_s_1  gap_ratio
symmetric   16 5.551115e-18       3.521806e-04 0.000214   4.297843
symmetric   32 5.551115e-18       8.800332e-05 0.000215   4.620198
symmetric   64 5.551115e-18       2.199136e-05 0.000216   4.734371
symmetric  128 0.000000e+00       5.496295e-06 0.000216   4.777856
symmetric  256 5.551115e-18       1.373036e-06 0.000216   4.796036
symmetric  512 1.110223e-17       3.422530e-07 0.000216   4.804221
symmetric 1024 0.000000e+00       8.455926e-08 0.000216   4.808085
     ramp   16 1.250000e-03       5.385933e-03 0.000181   4.568469
     ramp   32 6.250000e-04       2.682252e-03 0.000182   4.942375
     ramp   64 3.125000e-04       1.338423e-03 0.000183   5.140726
     ramp  128 1.562500e-04       6.685819e-04 0.000183   5.242880
     ramp  256 7.812500e-05       3.341403e-04 0.000183   5.294718
     ramp  512 3.906250e-05       1.670333e-04 0.000183   5.320830
     ramp 1024 1.953125e-05       

In [2]:
"""Redraw recorded numerical evidence as separate vector figures; no refitting."""
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
BASE=Path.cwd() / 'proton_R3_outputs'
OUT=BASE/'figures';OUT.mkdir(exist_ok=True)
d={'EXP_TIME_S': [0.00030539, 0.00033516000000000004, 0.00036784000000000003, 0.0004037, 0.00044306, 0.00048626, 0.00053367, 0.0005857, 0.00064281, 0.00070548, 0.0007742599999999999, 0.00084975, 0.0009326, 0.0010235300000000001, 0.0011233200000000001, 0.00123285, 0.0013530500000000002, 0.00148497, 0.0016297500000000001, 0.0017886500000000001, 0.00196304, 0.00215443, 0.00236449, 0.00259502, 0.00284804, 0.00312572, 0.00343047, 0.00376494, 0.00413201, 0.00497702, 0.00546228, 0.00599484, 0.0065793299999999996, 0.00722081, 0.00792483, 0.00869749, 0.00954548, 0.01047616, 0.01149757, 0.01261857, 0.013848860000000001, 0.01519911, 0.01668101, 0.018307379999999998, 0.020092330000000002, 0.02205131, 0.024201280000000002, 0.026560880000000002, 0.02915053, 0.03199267, 0.03511192, 0.03853529000000001, 0.042292430000000006, 0.04641589, 0.05094138, 0.055908099999999995, 0.06135907, 0.06734151000000001, 0.07390722, 0.08111308, 0.08902151000000001, 0.097701, 0.10722672, 0.1176812, 0.12915497, 0.14174742, 0.15556761, 0.17073526000000003, 0.18738174000000002, 0.20565123, 0.22570197, 0.24770764, 0.27185882, 0.29836471999999997, 0.32745492000000004, 0.35938137000000003, 0.39442061, 0.43287613, 0.47508102, 0.52140083, 0.57223677, 0.62802914, 0.6892612100000001, 0.7564633300000001, 0.8302175700000001, 0.9111627600000001, 1.0], 'HOLDOUT_Y': [0.99882, 1.0, 0.9949, 0.98685, 0.98799, 0.96335, 0.96316, 0.94221, 0.94022, 0.92379, 0.9182, 0.90521, 0.89902, 0.88021, 0.86898, 0.8697, 0.85976, 0.84735, 0.83184, 0.82131, 0.79909, 0.80459, 0.79284, 0.77442, 0.752, 0.76539, 0.72969, 0.74101, 0.73574, 0.7003, 0.66768, 0.68682, 0.64085, 0.66591, 0.61902, 0.63898, 0.60257, 0.60319, 0.60247, 0.56013, 0.55473, 0.55152, 0.53359, 0.52986, 0.51435, 0.49282, 0.47659, 0.49246, 0.46052, 0.45653, 0.44059, 0.43683, 0.4241, 0.41448, 0.39652, 0.37924, 0.37973, 0.36969, 0.3581, 0.34361, 0.34472, 0.33029, 0.30601, 0.31409, 0.30084, 0.28356, 0.2939, 0.26547, 0.29148, 0.26786, 0.23723, 0.23592, 0.22329, 0.22103, 0.21033, 0.19754, 0.19152, 0.19486, 0.18023, 0.17565, 0.17869, 0.16655, 0.14375, 0.14077, 0.1355, 0.14109, 0.12346], 'HOLDOUT_PI_LOW': [0.9019655815698713, 0.8848752465321755, 0.8660465539502816, 0.8701685504808965, 0.851967586675998, 0.8492138231276295, 0.8654673651876494, 0.8885401163452872, 0.8445580081741708, 0.838610846566655, 0.8748330373019545, 0.841289568741768, 0.8169548572355215, 0.8133115965450668, 0.8111649650183039, 0.8178646762433148, 0.8383420788447701, 0.7992603621339113, 0.7720964751542441, 0.7579902094922996, 0.770514529903368, 0.7642718269420615, 0.7511562473891634, 0.7139666323645322, 0.7019364895445427, 0.7287507755207792, 0.7009952137604651, 0.6724732167601026, 0.6557840734452139, 0.6303831329697612, 0.5909598699284285, 0.6084524923824741, 0.6124851516539424, 0.5538641413652372, 0.5370797778584092, 0.5723875868331285, 0.5591010432726118, 0.5205400583534904, 0.4715210006417836, 0.49214371933450407, 0.47419733729186375, 0.4524430562609749, 0.4844648518608508, 0.4312856689280934, 0.43260414488958976, 0.4313313507890276, 0.4186361789283446, 0.4049055965614271, 0.3805278658430661, 0.35614431378886147, 0.3545836828944896, 0.3265035958894043, 0.2946553768774011, 0.361214938552255, 0.3046345818896496, 0.28259501941777376, 0.29929086001297545, 0.2963950532007692, 0.25819890997243305, 0.24594994955541438, 0.24950686899413413, 0.22345280410041962, 0.23161307250429775, 0.20781749136340696, 0.16675317795131225, 0.19823180452302575, 0.17324110835680867, 0.21186572347621765, 0.16327475873785588, 0.1394529379298623, 0.15155999305042808, 0.17060964302989337, 0.10779501449963842, 0.07851054432528347, 0.11439950861040794, 0.1303717922793926, 0.0992273193369466, 0.0491878824114306, 0.05052015800507173, 0.09751979526256785, 0.05451779528831311, 0.036334175260702456, 0.055790381186998375, 0.022899252679896797, 0.016387003841349643, -0.022654016245932312, -0.04242587759547572], 'HOLDOUT_PI_HIGH': [1.1421108998031657, 1.076803272799995, 1.105538354642629, 1.0573580360974821, 1.1177477753048675, 1.0747465134116674, 1.081687312653359, 1.120912210661299, 1.0653570701296193, 1.0107448247231485, 1.0846849735678021, 1.097363607429991, 1.084214999789966, 1.060308062663197, 1.0216099358375421, 1.0466129388755239, 1.03080840456244, 1.0360031844827993, 1.0225399509591118, 1.0063529515358733, 0.9705638350856128, 0.9832339804411496, 0.9391649714885251, 0.9938700082669633, 0.9744482848358073, 0.9568837680401792, 0.9596210166384488, 0.9773120874607228, 0.9368767527896502, 0.8828921090229068, 0.8940769258166225, 0.887384931674202, 0.8329514916103372, 0.8264123352858666, 0.738486430471539, 0.7991766117278165, 0.7780892593456401, 0.7813803240859916, 0.7661215374252766, 0.7748077599399243, 0.7412818621002153, 0.670965254762527, 0.6919954080476322, 0.6548395979218185, 0.7192815857833191, 0.6797526087478201, 0.641843485952585, 0.6388529300796125, 0.587519399336673, 0.6245214258180922, 0.6200370391047023, 0.6171442060771026, 0.5245412186804153, 0.5884678955624397, 0.5450146701154345, 0.5439697404293011, 0.5024601512835384, 0.4718673898342285, 0.5224158919029893, 0.5192868115734293, 0.4774476715513306, 0.5166029819650496, 0.4888577036420434, 0.4611706974021585, 0.46244548559930254, 0.3808783390288959, 0.4548109379687102, 0.4957300337284087, 0.41920128859421174, 0.35870719460142064, 0.36916158085362344, 0.3204806454162509, 0.3447305928973438, 0.37657494503320743, 0.3623932971040794, 0.3474608525040995, 0.32031013403908515, 0.3042433723016063, 0.32486728912451956, 0.32600127826222636, 0.2673539142930066, 0.27171310572054846, 0.26633357752872433, 0.23260747592125558, 0.2222884238090052, 0.24523618216915893, 0.2341591582326225], 'strain_pct': [-3.0, 0.0, 3.0], 'K_R_001': [52.77777777777778, 36.66666666666667, 27.77777777777778], 'K_R_111': [66.11111111111111, 34.44444444444444, 28.88888888888889], 'TWIN_TEST_SNR': [8.831509238254943, 21.221268106346653, 16.530560362060953, 7.572623235789208, 17.199706954682874, 16.539667818319675, 4.6041800419426036, 10.18095472204863, 11.364360314161104, 17.061578061459585, 17.858227144737256, 14.402328225914088, 19.227924848734567, 10.280586002509724, 9.272143526849694, 9.678343852803312, 9.524936000440876, 24.034906060915546, 13.812540125828514, 8.729826181886427, 8.240170647728608, 20.673447873992192, 4.600732503810131, 8.853654526574083, 10.943897292720546, 13.812540125828514, 10.652819294566168, 22.791670160077523, 7.81850828043836, 19.452354828783204, 13.267589949099275, 17.275761573385694, 16.37437417922632, 13.733617746212634, 6.114368963545585, 10.144480181310632, 12.944891841225532, 15.84349424701225, 20.75434687656955, 8.586602237723207, 9.785490853521347, 13.733617746212634, 20.400880205803166, 18.79184585572229, 6.04172147034981, 17.238760214893915, 14.537123216225064, 10.742127077202026, 17.800275422248614, 9.083101798789924, 18.236929483133583, 9.005147991631228, 13.65427358447747, 18.880245558133545, 15.139147404216828, 7.63001386787727, 10.988782035277545, 11.66474773153895, 20.687814411826807, 14.387984155124572, 6.247151579366076, 13.24726385738296, 7.41772491122241, 8.073126170416357, 16.30915142253456, 9.662915653155853, 12.710611563495348, 19.227924848734688, 21.139422104562048, 18.717705342572643, 14.864940829026864, 7.731528110065318, 14.738058510994422, 20.961842962893243, 11.057090982342645, 17.011613119200494, 9.959929467783835, 10.972483440962003, 18.416720167770414, 12.744427761577931, 13.047321138027842, 22.30667685407994, 7.5620723754900405, 16.425637118812432, 10.936665915418304, 6.947037560021315, 9.659228537321152, 9.20793429698841, 14.38767020019602, 24.102861674433907, 16.99257034877043, 5.947569625547007, 7.212922636171681, 7.490065880263526, 18.26919522131097, 14.315255925384532, 19.5297664864811, 13.819329110945352, 9.456282120995665, 16.08728043746887, 18.40134907458443, 18.627237167929778, 10.168489250796862, 8.745141003885815, 12.744427761577818, 9.208360083885207, 13.384006112447352, 7.102402106551665, 14.533873577874346, 13.701605132279699, 24.215953405192117, 16.458174227259487, 8.222525518260237, 18.416720167770414, 7.022878581473812, 8.89069697400616, 8.620592778235125, 7.325854690185031, 16.964501493049347, 9.460039464969052, 8.248033152849288, 5.948217911151068, 7.6017934652164705, 9.08310179878966, 11.358934853684325, 10.894245630153781, 16.87481933443969, 15.866079339462242, 9.910999636386903, 4.699888776281281, 17.663018476510608, 9.772420069565516, 24.03490606091835, 8.587814458052078, 11.353877248488404, 14.453217194088841, 8.66531476789001, 10.943897292719496, 23.785113570699128, 20.932788882229303, 23.397131678214716, 6.779024911198955, 13.25253093670053, 4.791134588801436, 16.231243634144622, 18.3422446864282, 18.416720167773317, 4.6470870698915006, 23.020900209712117, 9.963792639780197, 15.771393599326093, 8.968224121990586, 7.968001246704524, 18.717705342572643, 9.971222117591125, 18.957939006673275, 9.850072356067669, 16.720460251319317, 10.88330680013106, 9.613962424366218, 20.255268867104956, 9.95881717485339, 13.312689754973839, 8.619380107447073, 20.72505672881731, 12.435034037289965, 9.208360083886658, 24.076460822854308, 21.54845026861789, 8.83150923825487, 9.528030650240778, 10.463849667611559, 18.973093780643143, 12.744427761577818, 11.243147454274617, 8.460166841837717, 8.371992738937758, 21.548450268618996, 10.728006599109149, 7.3318044241547735, 10.060888179705172, 14.038279006927965, 7.122964806191256, 9.234252671671461, 7.795911823347683, 8.341297232011334, 17.834281061641438, 12.929070161171419, 7.177774509970013, 13.247263857382437, 13.29801286625809, 12.603453959150741, 20.58135259588819, 19.42022978850305, 16.320704164639377, 20.104397371743783, 4.6041800419426036, 7.572623235789592, 15.357515568178867, 11.855269460356844, 9.15306495078941, 11.355767009772187, 21.240712935960083, 16.832008908065674, 7.464889642904856, 8.831509238255304, 8.887216085954579, 17.70100977025106, 7.3715619065447635, 18.38278506923873, 7.8348075863575755, 22.17560369004274, 11.369371735117836, 4.576532475394705, 19.529766486477463, 23.77440558051091, 11.997836267997153, 13.97647796237281, 15.352277988337422, 18.077931534264426, 12.075102674334993, 23.397131678212165, 8.167538760883891, 13.83532693706962, 8.496285174385568, 16.34404653086911, 4.504743218738305, 20.039724148033205, 13.214666181849896, 12.55967332933822, 18.15071343627089, 7.5047383164092185, 4.752081617541546, 12.200787562078322, 6.15147997162814, 13.404235238629132, 11.164534401433308, 16.33813403171766, 8.85365452657493, 8.73074919281111, 9.311767015187655, 6.232797262695938, 18.166801980409762, 22.413968759389782, 7.998557511998106, 5.84852881979254, 12.68130434790383, 17.849685864152217, 9.577206983647335, 18.536713878673005, 9.709043969844805, 16.01697402265787, 20.85967022218285, 13.38400611244783, 8.588003196362862, 7.780941931513278, 11.24537065048388, 9.627108124087435, 11.682211259023653, 21.240712935964158, 8.72449651529698, 9.910999636386903, 22.227526530973726, 17.23876021489372, 13.981595690686744, 14.756090877625054, 11.844671865872812, 7.62910849393389, 23.201984362882463, 15.702206196102832, 21.54845026861917, 7.614087912760599, 17.16544754302656, 10.986894196971788, 16.853149682256475, 7.495453231283994, 11.07888769085903], 'TWIN_SYMMETRIC_SNR': [0.28244872292093764, 0.6777406508719838, 0.5289945176306414, 0.24741891217399165, 0.5495631425263572, 0.527717506322473, 0.14746070299093794, 0.3312104485696641, 0.3613537658851394, 0.5449914050194351, 0.5713584538999636, 0.46190235061889506, 0.6166912808370186, 0.3254817194849956, 0.2921021786433437, 0.3104764736719129, 0.30539913235854677, 0.7708641010461862, 0.44238210897282937, 0.2790833124647967, 0.25793561920906066, 0.6596091261981603, 0.14734661391622283, 0.27824294599014604, 0.33753054923334685, 0.44238210897282937, 0.3329752580249816, 0.7297167410627742, 0.24892091316834772, 0.6191938813084304, 0.42434581135494454, 0.5471471892020788, 0.5416496604206207, 0.42989269868171576, 0.20568653110702423, 0.33087642423422764, 0.41366539430196875, 0.5046758932364999, 0.6622869374479727, 0.29194151082175285, 0.3288999920332408, 0.42989269868171576, 0.6505878420198861, 0.6022583641281841, 0.19667863904796237, 0.5508556391287684, 0.4663636581849352, 0.34079710824822346, 0.5694403523268541, 0.2809307387069707, 0.5838923033005226, 0.2881956757659172, 0.458671747253127, 0.6002574517163398, 0.4813632297585348, 0.25311499729871684, 0.3489227871512286, 0.37129522773984874, 0.6600846749864728, 0.4614276000526567, 0.21009221353868832, 0.4236730843814169, 0.26398496639132724, 0.2524107037174936, 0.5481666533887138, 0.324833010319551, 0.396028072611477, 0.6166912808370238, 0.6750318369959076, 0.5998045331889242, 0.4722859915953013, 0.24604171108233597, 0.468088345603025, 0.6691543398671812, 0.3474940360848521, 0.5433376501627956, 0.3103943632408701, 0.3483833401999519, 0.5898428119637518, 0.4070305343747503, 0.4385333227109721, 0.7136647838164241, 0.23549455208586212, 0.5190231870457649, 0.3471978603555066, 0.24836587113196082, 0.3098438137476735, 0.32785009589685976, 0.456537377685556, 0.773113229197437, 0.542707379166325, 0.20015216567446703, 0.2389075238596936, 0.23805021357180375, 0.580033966639354, 0.4590205125544156, 0.6266813386163997, 0.46414821642576704, 0.29819222078193164, 0.5127442226357646, 0.5893340740985911, 0.5968103044030133, 0.31682245808919823, 0.28888729641532346, 0.4070305343747399, 0.2949214059818759, 0.42819887028983156, 0.2535214300541699, 0.4613302818148749, 0.43378564660996966, 0.7768562281230392, 0.5250202036884483, 0.2906911468264887, 0.5898428119637518, 0.21763665884025934, 0.2844076876369667, 0.3016264597793717, 0.23261519143415413, 0.5368441773882817, 0.2883924745423355, 0.27361835946952734, 0.20017400672147143, 0.24174826122819607, 0.2809307387069551, 0.3711283682610005, 0.34579384397720075, 0.5338804529031955, 0.5054232794887873, 0.3132447094675801, 0.15062837938902426, 0.5648974458419013, 0.3387240017246373, 0.7708641010462798, 0.2743830698755022, 0.3511634233837394, 0.4586607362005973, 0.267051762927118, 0.33753054923331566, 0.7625967141657342, 0.6681927059920978, 0.749755666486115, 0.24279062743218502, 0.4238474553608275, 0.15364834556914952, 0.5369020867078808, 0.5873778930371086, 0.5898428119638401, 0.1488808000326481, 0.7373035149546533, 0.33003992340957833, 0.5022898043645853, 0.2900971088228556, 0.2643316341996189, 0.5998045331889242, 0.30550608638973886, 0.6194216594821879, 0.3267675368257022, 0.5337011838453004, 0.3454421411618987, 0.3083456404184755, 0.645768553459824, 0.3294409044709331, 0.42583851720903204, 0.2754278195643842, 0.6613173880552512, 0.3967904328331351, 0.29492140598192007, 0.7722394397373062, 0.688569548910971, 0.28244872292093764, 0.30550155483425423, 0.35140765979876304, 0.6033317143548278, 0.4070305343747399, 0.3573417123229637, 0.2652184183026446, 0.267239934487946, 0.6885695489110126, 0.3365872993112876, 0.2179038300116892, 0.3107683245318454, 0.4498533998916399, 0.23537648759618848, 0.31201829428340144, 0.2481729338415121, 0.27549138825129854, 0.5705657971668426, 0.41314172934659715, 0.24901302476158207, 0.42367308438141166, 0.4253527635224681, 0.39249092014311393, 0.6565611263246498, 0.6230560007146295, 0.5204702736158111, 0.6408099232273257, 0.14746070299093794, 0.24741891217400466, 0.4948655817230394, 0.3776099558178866, 0.2930912994932093, 0.3523273586019791, 0.6783842239578062, 0.5373932263085488, 0.234075860784, 0.28244872292095063, 0.28429248039479443, 0.5612492758482026, 0.23412799785936653, 0.5887203448799908, 0.2494604609737816, 0.709326659839409, 0.36550555551373903, 0.14654564974660464, 0.6266813386162853, 0.7622424032273173, 0.3823204344794584, 0.447808082745651, 0.5150107547906694, 0.5786301098978293, 0.40205333645316454, 0.7497556664860319, 0.27803249731170576, 0.4577267856370192, 0.2713536895831729, 0.5212429235773999, 0.14418783555805145, 0.638634461620555, 0.4176596126234644, 0.40091562359528365, 0.5810388633344179, 0.23853583848454304, 0.1523558073516716, 0.38903740740886383, 0.20344943532841298, 0.4288684253771578, 0.3547397965640133, 0.5210474658161304, 0.27824294599018246, 0.2791138410664782, 0.2934087273593964, 0.19643222746521818, 0.5766674225459085, 0.7172158828366199, 0.25488028965297227, 0.1968660679219384, 0.40494142089958024, 0.5710756081153907, 0.30712914492834403, 0.5938142512860024, 0.3146524470384883, 0.5104173376693899, 0.6657727022637978, 0.42819887028984716, 0.2598260215694655, 0.2476775525233727, 0.35741980022078546, 0.3250514732579611, 0.3669398779233404, 0.678384223957931, 0.2789069369532158, 0.3132447094675801, 0.7110451482785399, 0.5508556391287581, 0.44110137665846383, 0.46373824331697416, 0.3772508058345912, 0.24265221082180663, 0.7432968688243904, 0.49509729579464595, 0.6885695489110022, 0.2421577505656889, 0.5484292286165288, 0.3439141589454246, 0.53315772548277, 0.23054366773832766, 0.3444136853452311], 'CHI_PROFILE_GRID': [0.4698094585796684, 0.5031427919130017, 0.536476125246335, 0.5698094585796684, 0.6031427919130017, 0.636476125246335, 0.6698094585796683, 0.7031427919130018, 0.736476125246335, 0.7698094585796684, 0.8031427919130016, 0.8364761252463351, 0.8698094585796684, 0.9031427919130017, 0.9364761252463351, 0.9698094585796684, 1.0031427919130018, 1.036476125246335, 1.0698094585796682, 1.1031427919130017, 1.1364761252463351, 1.1698094585796683, 1.2031427919130016, 1.236476125246335, 1.2698094585796684, 1.3031427919130016, 1.336476125246335, 1.3698094585796685, 1.4031427919130017, 1.436476125246335, 1.4698094585796684], 'CHI_PROFILE_SSE': [1.7253671976003142, 1.4503515931854736, 1.2268304870316673, 1.0583682179804326, 0.9475067803544369, 0.8951573549091247, 0.86530422259906, 0.8385505558830542, 0.8157419089635001, 0.7970118964197889, 0.7821987605079549, 0.7709940858569693, 0.7630134124615425, 0.7578369902084985, 0.7550363645429902, 0.7541926124280254, 0.754908236072467, 0.7568121028237887, 0.7595519751249814, 0.7627641915363128, 0.7660913728598664, 0.7693667355038872, 0.7725927354462985, 0.7804187920021193, 0.7834031835022026, 0.785845176465593, 0.7878633516192247, 0.7895284934298973, 0.7908945151830515, 0.7920061239719928, 0.7929018616539735], 'models': {'Single exponential': [0.993576174575632, 0.9929521794820286, 0.9922676404452528, 0.9915170338403482, 0.9906938203078196, 0.9897910799910792, 0.9888013110078828, 0.9877162303011106, 0.9865265774261626, 0.9852227535796502, 0.9837937969483864, 0.9822278205630096, 0.9805120361029268, 0.9786323683438092, 0.976573697165797, 0.9743190744398657, 0.9718508049900793, 0.9691490655982182, 0.9661925956668778, 0.9629581694679132, 0.9594209035958976, 0.9555537691350794, 0.9513273374640762, 0.9467105587454158, 0.9416691636534836, 0.936167314038156, 0.9301661011174854, 0.923623906034438, 0.9164970059956916, 0.9002989061728978, 0.8911266316160297, 0.8811678218255922, 0.8703659830872104, 0.8586632737913188, 0.8460006598055455, 0.8323183034893872, 0.8175565573215774, 0.8016565248224639, 0.7845621221880348, 0.7662202441778829, 0.7465833305994933, 0.7256104840779266, 0.7032702826786051, 0.6795430079483695, 0.6544225862707724, 0.6279205050152749, 0.6000682798208303, 0.5709202703592939, 0.5405575014040878, 0.5090897109313531, 0.4766582230554255, 0.4434377557711206, 0.4096371331570886, 0.3754989504589053, 0.3412979043641295, 0.3073369247314468, 0.273941492675801, 0.2414516015594535, 0.2102116705963961, 0.1805579956258987, 0.1528050576571093, 0.1272307655550801, 0.1040617302159339, 0.083459939707639, 0.0655124023478593, 0.0502246051962618, 0.0375195460027928, 0.0272426560693476, 0.0191729161799894, 0.0130392428892051, 0.0085406675718412, 0.0053680278945057, 0.003224590642881, 0.0018431213575386, 0.0009975823142721, 0.0005085690799548, 0.0002427857193554, 0.0001078417383543, 4.4257663505241885e-05, 1.6652435558769552e-05, 5.696101472849605e-06, 1.7549029271136668e-06, 4.820308598448247e-07, 1.1672998011720303e-07, 2.4617347870645072e-08, 4.4606347372550075e-09, 6.842632009196646e-10], 'Stretched exponential': [0.8980598587994858, 0.8944968961902411, 0.890816258201915, 0.8870161648921513, 0.8830922872862267, 0.8790414150983538, 0.8748605462922283, 0.8705462677403014, 0.866094369642169, 0.8615024870797513, 0.8567665148606464, 0.8518826394586078, 0.8468475756762334, 0.8416576255853035, 0.8363096422190259, 0.8307992562474789, 0.8251238774557449, 0.8192795393692354, 0.8132628781740535, 0.807070225569128, 0.8006984639222658, 0.7941442941564425, 0.787404103395486, 0.7804753954755786, 0.7733544756846691, 0.7660390858713049, 0.7585263356793023, 0.7508134100175731, 0.7428983080390548, 0.7264521229547356, 0.7179173718972951, 0.7091730983851359, 0.7002178427968684, 0.6910508040087973, 0.6816715529575881, 0.6720799018457106, 0.6622761111751576, 0.6522606691403253, 0.6420349016029981, 0.631600240003471, 0.6209588396446035, 0.6101131796079065, 0.5990664636407688, 0.5878225353398508, 0.5763855811033004, 0.5647606613229648, 0.55295346667579, 0.5409702384516643, 0.5288181178920316, 0.5165048384439122, 0.5040389248609449, 0.4914297081194145, 0.4786872718193191, 0.4658224617460011, 0.4528469633221029, 0.4397732355154743, 0.4266145391579478, 0.413384911373053, 0.4000992132156291, 0.3867729994450753, 0.3734225691126167, 0.3600649438734958, 0.3467177951871386, 0.333399363975766, 0.3201285057623633, 0.3069245272161608, 0.2938071779567939, 0.2807965253245292, 0.2679129128381073, 0.2551768417915944, 0.2426088687242441, 0.2302294915675687, 0.2180590576394698, 0.206117598733886, 0.194424742724131, 0.1829995716649406, 0.1718604734438526, 0.1610250196440878, 0.15050982224581, 0.1403304055408208, 0.1305010664867787, 0.1210347578584715, 0.1119429569296452, 0.1032355683805914, 0.0949208170271505, 0.0870051629414169, 0.0794932328383566], 'Distributed relaxation': [0.9735994803927548, 0.971129694569665, 0.968438945571644, 0.9655107634998256, 0.9623258585270712, 0.9588648437775896, 0.9551076947553918, 0.9510332890608268, 0.9466190411699216, 0.941843673710615, 0.936683870885981, 0.931116455426588, 0.9251190386594852, 0.91866930937722, 0.911746605124382, 0.9043300869552788, 0.8964032201730748, 0.8879501594590469, 0.8789590418466271, 0.8694213777471145, 0.8593340392008871, 0.8486988479154481, 0.837522953678049, 0.8258217896301536, 0.8136156710281243, 0.8009345241580647, 0.7878144322520493, 0.7742986091815175, 0.7604381702826196, 0.7319120835165202, 0.7173718946548465, 0.7027344931960572, 0.6880643222174018, 0.6734235775560985, 0.6588695991359306, 0.6444527935564432, 0.6302154739131821, 0.6161905755792065, 0.6024021444946462, 0.5888645486966539, 0.5755841208549014, 0.5625598465141841, 0.5497853241551369, 0.5372501491398848, 0.5249409907165024, 0.5128433509338803, 0.5009421558022457, 0.4892222717463879, 0.4776691968228626, 0.4662689241279956, 0.4550081967782892, 0.4438745183478717, 0.4328560819916605, 0.4219417546918318, 0.4111211205001052, 0.400384408194147, 0.3897225088725501, 0.3791269576280144, 0.3685899835374115, 0.3581044278127869, 0.3476638170500375, 0.3372623882856217, 0.3268950854616724, 0.3165575616330923, 0.3062462896084507, 0.2959585220293095, 0.2856923834099372, 0.2754468822406307, 0.2652219993924877, 0.255018726951733, 0.2448391289632507, 0.234686401693382, 0.2245649594582754, 0.2144804661488003, 0.2044399333854175, 0.1944517759162894, 0.1845258583726991, 0.1746735587763655, 0.1649078033530116, 0.1552430996345017, 0.1456955376499523, 0.1362827849523627, 0.1270240327514368, 0.1179399332381468, 0.1090524763397954, 0.1003848299868419, 0.0919611304731821], 'Biexponential (six parameters)': [0.9769955005284858, 0.9748037484475844, 0.9724079641972844, 0.969791293902676, 0.96693390436357, 0.9638153246401252, 0.9604138756971126, 0.9567061476531404, 0.9526665347944872, 0.9482696345864114, 0.9434870367999189, 0.9382891422778185, 0.9326456012681148, 0.9265244658784804, 0.9198934841596716, 0.912718159790468, 0.9049658866737466, 0.8966023140346047, 0.8875944327945939, 0.877909938556118, 0.867519285764489, 0.8563954662099935, 0.8445147089153426, 0.8318601770286871, 0.8184191507536502, 0.8041892762355973, 0.7891762903095867, 0.773396900977162, 0.7568819859910884, 0.7218381537956631, 0.703447914738029, 0.6846016091930385, 0.6654134158649648, 0.6460165191265175, 0.6265612185199111, 0.6072123517036526, 0.5881462583837422, 0.5695456530409595, 0.5515948278301095, 0.5344712454151379, 0.518338782815527, 0.5033387915975409, 0.4895826065093308, 0.4771441200808795, 0.466053909541656, 0.4562966692363681, 0.4478107654582267, 0.4404914558047202, 0.4341976953281788, 0.4287613498488299, 0.423998642153242, 0.4197220837222356, 0.4157515785690374, 0.4119234183187777, 0.4080962113817954, 0.4041533191764491, 0.400002132927727, 0.3955709332836362, 0.390804452629242, 0.3856591310582707, 0.3800990173364874, 0.374092692634313, 0.367611311702414, 0.3606276217270144, 0.353115779392022, 0.3450515501201012, 0.3364128977098766, 0.327180713663501, 0.3173397563688926, 0.3068796759417418, 0.2957961691196631, 0.2840922312899679, 0.2717795146995649, 0.2588796653430351, 0.2454257332845281, 0.2314634659177932, 0.2170524374145389, 0.2022669542826515, 0.1871965598089057, 0.1719460475458356, 0.1566348047180147, 0.1413953960719181, 0.1263712102265648, 0.1117131600937414, 0.0975753409425419, 0.0841097559576072, 0.0714602640480801]}, 'shift': [{'split': 'random', 'n train': 1680, 'n test': 720, 'test failure prevalence': 0.2, 'visible-model RMSE': 0.0017201552537285, 'ridge alpha': 1e-10, 'baseline selected on train': 'gradient L2', 'Δ ROC-AUC': 0.7617066936728395, 'baseline ROC-AUC': 0.7796344521604939, 'Δ-baseline AUC': -0.0179277584876543, 'difference 2.5%': -0.0409571675738676, 'difference 97.5%': 0.0059844047048639}, {'split': 'concentration holdout x_H=0.50', 'n train': 1944, 'n test': 456, 'test failure prevalence': 0.25, 'visible-model RMSE': 0.0024743914553989, 'ridge alpha': 1e-10, 'baseline selected on train': 'gradient L2', 'Δ ROC-AUC': 0.8307684415717657, 'baseline ROC-AUC': 0.8569559864573715, 'Δ-baseline AUC': -0.0261875448856058, 'difference 2.5%': -0.0492106925267795, 'difference 97.5%': -0.0058653226239741}, {'split': 'temperature holdout 350 K', 'n train': 1794, 'n test': 606, 'test failure prevalence': 0.504950495049505, 'visible-model RMSE': 0.0030797527595565, 'ridge alpha': 0.1, 'baseline selected on train': 'spatial variance', 'Δ ROC-AUC': 0.6450217864923747, 'baseline ROC-AUC': 0.6454030501089325, 'Δ-baseline AUC': -0.0003812636165577, 'difference 2.5%': -0.0079154475562404, 'difference 97.5%': 0.0067957984384023}, {'split': 'field holdout 3e5 V/m', 'n train': 1582, 'n test': 818, 'test failure prevalence': 0.5635696821515892, 'visible-model RMSE': 0.0031753450630379, 'ridge alpha': 0.1, 'baseline selected on train': 'gradient L2', 'Δ ROC-AUC': 0.7793130267291298, 'baseline ROC-AUC': 0.7339482430716321, 'Δ-baseline AUC': 0.0453647836574977, 'difference 2.5%': 0.0252764264989931, 'difference 97.5%': 0.0652427787917036}, {'split': 'protocol holdout mixed', 'n train': 1959, 'n test': 441, 'test failure prevalence': 0.1700680272108843, 'visible-model RMSE': 0.001630099050327, 'ridge alpha': 0.0251188643150958, 'baseline selected on train': 'gradient L2', 'Δ ROC-AUC': 0.8703825136612022, 'baseline ROC-AUC': 0.867431693989071, 'Δ-baseline AUC': 0.0029508196721311, 'difference 2.5%': -0.0222363387641164, 'difference 97.5%': 0.0279168465409252}]}
r=pd.read_csv(BASE/'new_results/full_generator_refinement.csv')
plt.rcParams.update({'font.size':11,'axes.labelsize':11,'xtick.labelsize':10,'ytick.labelsize':10,'legend.fontsize':10,'pdf.fonttype':42,'svg.fonttype':'none'})

def save(fig,name):
 fig.tight_layout()
 fig.savefig(OUT/(name+'.pdf'))
 fig.savefig(OUT/(name+'.svg'))
 fig.savefig(OUT/(name+'.png'),dpi=180)
 plt.close(fig)

fig,ax=plt.subplots(figsize=(6.4,4.7))
t=np.array(d['EXP_TIME_S'])
ax.fill_between(t,d['HOLDOUT_PI_LOW'],d['HOLDOUT_PI_HIGH'],alpha=.17,label='Calibration-only 95% interval')
ax.scatter(t,d['HOLDOUT_Y'],s=14,marker='o',label='Published 533 kV/cm trace',zorder=5)
labels={'Single exponential':'Single exponential (2)', 'Stretched exponential':'Stretched exponential (4)',
'Distributed relaxation':'Distributed relaxation (6)','Biexponential (six parameters)':'Biexponential (6; retrospective)'}
for (m,p),ls in zip(d['models'].items(),[':', '--','-', '-.']):
 ax.plot(t,p,ls=ls,lw=1.7,label=labels[m])
ax.set_xscale('log');ax.set_xlabel('Time (s)');ax.set_ylabel('Normalized polarization')
ax.set_ylim(-.08,1.16);ax.legend(loc='upper center',bbox_to_anchor=(.5,1.37),ncol=2,frameon=False)
save(fig,'proton_holdout')

fig,ax=plt.subplots(figsize=(6.4,4.3))
for case,mark,ls,label in [('symmetric','o','-','Symmetric background'),('ramp','s','--','Gradient background')]:
 g=r[r.case==case];ax.loglog(g.N,g.relative_L2_error,marker=mark,ls=ls,label=label)
ns=np.array([64,1024]);err=r[(r.case=='symmetric') & (r.N==64)].relative_L2_error.iloc[0]
ax.loglog(ns,err*(64/ns)**2*.55,ls=':',label=r'$N^{-2}$ guide')
err=r[(r.case=='ramp') & (r.N==64)].relative_L2_error.iloc[0]
ax.loglog(ns,err*64/ns*1.5,ls='-.',label=r'$N^{-1}$ guide')
ax.set_xlabel('Number of coarse cells, N');ax.set_ylabel('Relative spatial error');ax.legend(frameon=False)
save(fig,'proton_full_convergence')

fig,ax=plt.subplots(figsize=(6.4,4.1))
for case,mark,ls,label in [('symmetric','o','-','Symmetric background'),('ramp','s','--','Gradient background')]:
 g=r[r.case==case];ax.semilogx(g.N,g.gap_s_1,marker=mark,ls=ls,label=label)
bound=json.loads((BASE/'new_results/summary.json').read_text())['uniform_bound_s_1']
ax.axhline(bound,ls=':',label='Common mesh-independent lower bound')
ax.set_xlabel('Number of coarse cells, N');ax.set_ylabel(r'Frozen-generator spectral gap (s$^{-1}$)')
ax.set_ylim(0,2.4e-4);ax.ticklabel_format(axis='y',style='sci',scilimits=(0,0));ax.legend(frameon=False,loc='upper center',bbox_to_anchor=(.5,1.32))
save(fig,'proton_full_gaps')

fig,ax=plt.subplots(figsize=(6.4,3.7))
for name in ['Distributed relaxation','Biexponential (six parameters)']:
 ax.semilogx(t,np.asarray(d['models'][name])-d['HOLDOUT_Y'],marker='.',ms=4,label=labels[name])
ax.axhline(0,ls=':');ax.set_xlabel('Time (s)');ax.set_ylabel('Prediction minus observation');ax.legend(frameon=False,loc='upper center',bbox_to_anchor=(.5,1.26))
save(fig,'supp_polarization_residuals')

fig,ax=plt.subplots(figsize=(6.4,3.9))
for ori,ls,m in [('001','-','o'),('111','--','s')]:
 x=np.array(d['strain_pct']);y=np.array(d['K_R_'+ori]);ax.scatter(x,y,marker=m,label=f'[{ori}] curvature inputs')
 xx=np.linspace(-3,3,120);ax.plot(xx,np.exp(np.polyval(np.polyfit(x,np.log(y),1),xx)),ls=ls,label=f'[{ori}] log-linear fit')
ax.set_xlabel('Biaxial strain (%)');ax.set_ylabel(r'Breathing stiffness (eV $\AA^{-2}$)');ax.legend(frameon=False,ncol=2,loc='upper center',bbox_to_anchor=(.5,1.29))
save(fig,'supp_structural_curvature')

fig,ax=plt.subplots(figsize=(6.4,3.9))
sh=d['shift'];x=np.arange(5)
ax.plot(x,[v['Δ ROC-AUC'] for v in sh],'o-',label='Recoverability')
ax.plot(x,[v['baseline ROC-AUC'] for v in sh],'s--',label='Comparator selected on training set')
ax.axhline(.5,ls=':');ax.set_xticks(x,['Random','Concentration','Temperature','Field','Protocol'])
ax.set_ylabel('Receiver operating characteristic area');ax.set_ylim(.45,1.);ax.legend(frameon=False,loc='upper center',bbox_to_anchor=(.5,1.25))
save(fig,'supp_recoverability_transfer')

fig,ax=plt.subplots(figsize=(6.4,4.0))
for key,label,ls in [('TWIN_TEST_SNR','Right-electrode continuation','-'),('TWIN_SYMMETRIC_SNR','Symmetric-control continuation','--')]:
 v=np.sort(d[key]);ax.step(v,np.arange(1,len(v)+1)/len(v),where='post',label=label,ls=ls)
ax.axvline(3,ls=':',label='Effect-size reference: 3');ax.set_xscale('log');ax.set_xlabel('Twin separation / published residual-scale proxy')
ax.set_ylabel('Empirical cumulative fraction');ax.set_ylim(0,1.02);ax.legend(frameon=False,loc='upper center',bbox_to_anchor=(.5,1.33))
save(fig,'supp_hidden_twins')
print('Created seven single-axis vector figures in',OUT)

(BASE/"plot_data.json").write_text(json.dumps(d,indent=2))


Created seven single-axis vector figures in /mnt/data/proton_R3_review/deliverables/proton_R3_outputs/figures


33831